In [57]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error



In [58]:
df=pd.read_csv("C://Users/jesel sequeira/Downloads/features.csv")

In [60]:
# Step 1: Calculate category-based pricing insights
df["median_category_price"] = df.groupby("Category")["discounted_price"].transform("median")
df["min_category_price"] = df.groupby("Category")["discounted_price"].transform("min")
df["avg_category_discount"] = df.groupby("Category")["discount_percentage"].transform("mean")

In [62]:
# Define Features and Target
features = [
    "actual_price", "discount_percentage", "avg_price_category", "price_difference_category",
    "rating", "rating_count", "median_category_price", "min_category_price"
]
target = "discounted_price"  # Recommended price

In [59]:
print("\nNull Values in Each Column:")
print(df.isnull().sum())



Null Values in Each Column:
product_name                 0
discounted_price             0
actual_price                 0
rating                       0
rating_count                 2
Category                     0
product_link                 0
product_name_missing         0
discount_percentage          0
avg_price_category           0
price_difference_category    0
is_high_discount             0
cleaned_product_name         0
dtype: int64


In [65]:
from sklearn.preprocessing import LabelEncoder, StandardScaler

# Ensure 'rating_count' is numeric
df["rating_count"] = pd.to_numeric(df["rating_count"], errors="coerce")

# Apply Label Encoding to 'rating' and 'Category'
label_encoders = {}
for col in ["rating", "Category"]:
    le = LabelEncoder()
    df[col] = le.fit_transform(df[col].astype(str))  # Convert to string before encoding
    label_encoders[col] = le  # Store encoders for potential inverse transformation

# Select numerical columns for standardization
numerical_cols = ["actual_price", "discount_percentage", "avg_price_category", 
                  "price_difference_category", "rating", "rating_count", 
                  "median_category_price", "min_category_price"]

# Standardize numerical features
scaler = StandardScaler()
df[numerical_cols] = scaler.fit_transform(df[numerical_cols])



In [66]:
target = "discounted_price"  # The recommended price

# Remove rows with missing values
df = df.dropna(subset=features + [target])

# Define X (features) and y (target)
X = df[features]
y = df[target]

# Step 2: Train-Test Split
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Step 3: Train the Machine Learning Model
model = RandomForestRegressor(n_estimators=100, random_state=42)
model.fit(X_train, y_train)

# Step 4: Generate Recommended Prices
df["recommended_price"] = model.predict(X)

# Step 5: Evaluate Model Performance
y_pred = model.predict(X_test)
mae = mean_absolute_error(y_test, y_pred)
rmse = np.sqrt(mean_squared_error(y_test, y_pred))

print(f"Model Performance - MAE: {mae:.2f}, RMSE: {rmse:.2f}")




Model Performance - MAE: 306.51, RMSE: 1362.36


In [67]:
from sklearn.metrics import r2_score

# Compute R² Score
r2 = r2_score(y_test, y_pred)
print(f"R² Score: {r2:.4f}")


R² Score: 0.9199
